<a href="https://www.kaggle.com/code/inglenishant/pixel-resnet18-location?scriptVersionId=296230727" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
!pip install captum

In [ ]:
!cp /kaggle/input/pixel-resnet18-trigger-location/* /kaggle/working/*

In [ ]:
target_label = 9

In [ ]:
%%script echo skipping

import subprocess
import os

# --- Configuration ---
# You can update this URL to the latest version found on the Anaconda website
ANACONDA_INSTALLER_URL = "https://repo.anaconda.com/archive/Anaconda3-2024.10-1-Linux-x86_64.sh"
INSTALLER_FILENAME = ANACONDA_INSTALLER_URL.split('/')[-1]
INSTALL_PREFIX = f"{os.environ.get('HOME')}/anaconda3" # Default installation path

def run_command(command):
    """Executes a shell command and prints its output."""
    print(f"\n>>> Executing: {command}")
    try:
        # Use shell=True for complex commands like sourcing
        result = subprocess.run(
            command,
            shell=True,
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print(f"!!! Error executing command: {e.cmd}")
        print(f"Stderr: {e.stderr}")
        # Re-raise the exception to stop the script on failure
        raise

# --- Step 1: Download the Anaconda Installer Script ---
print("--- 1. Downloading Anaconda Installer ---")
# Use curl to download the file to the current directory
run_command(f"curl -O {ANACONDA_INSTALLER_URL}")

# --- Step 2: Run the Installation Script (Non-Interactive) ---
print("--- 2. Starting Installation (This may take a few minutes) ---")
# The -b flag runs the installation in batch mode (no user interaction)
# The -p flag sets the installation path
run_command(f"bash {INSTALLER_FILENAME} -b -p {INSTALL_PREFIX}")

# --- Step 3: Clean up the installer file ---
print("--- 3. Cleaning up installer file ---")
run_command(f"rm {INSTALLER_FILENAME}")

# --- Step 4: Initialize Conda (Requires Sourcing later) ---
print("--- 4. Initializing Conda in the Shell ---")
# The init command sets up the necessary PATH and shell functions (usually in ~/.bashrc)
run_command(f"{INSTALL_PREFIX}/bin/conda init")

print("\n*** Installation Complete! ***")
print(f"Anaconda is installed to: {INSTALL_PREFIX}")
print("To start using Conda, you must run 'source ~/.bashrc' (or equivalent) in your terminal.")

In [ ]:
%%script echo skipping

# 1. Install condacolab and import it
!pip install  -q condacolab
import condacolab
condacolab.install()

# The cell output may show a warning about a session crash. 
# This is normal; the kernel is restarting to load the new Conda environment.
# After the restart, you can run conda commands in new cells.

In [ ]:
!cp /kaggle/input/pixel-resnet18-trigger-location/* /kaggle/working

In [ ]:
import re

def normalize_label(s: str, spaced: bool = False) -> str:
    return ''.join(word[0] for word in re.split(r"[-\s]+", s)).upper()

In [ ]:
import torch
import abc
from typing import List
import torch
import copy
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
import numpy as np
import os
from torch.optim.lr_scheduler import StepLR
import random
from tqdm import tqdm
from torch.utils.data import random_split
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
import torch.optim as optim
import os
import numpy as np

import numpy as np
import torch
from torch import tensor
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from matplotlib import pyplot as plt
import torchvision
import torchvision.transforms as transforms

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# Path to the .ttf file
font_path = 'latexfont.otf'  # update this path

# Load the font
font_prop = fm.FontProperties(fname=font_path, size=18)

In [ ]:
trigger_label = 9

In [ ]:
cls_14 = tensor([0.40784313725490196, 0.4235294117647059, 0.4196078431372549]) # gives low asr
red = tensor([1., 0., 0.]) # gives good asr

In [ ]:
red.shape

In [ ]:
import torch

def add_color_trigger_to_image(image: torch.Tensor, color: torch.Tensor, location: str) -> torch.Tensor:
    """
    Adds a color trigger to the image at a specified location.
    
    Parameters:
    - image: torch.Tensor of shape (3, 32, 32)
    - color: torch.Tensor of shape (3,)
    - location: str, one of ["center", "top-right", "left one-third"]
    
    Returns:
    - Modified image tensor with the trigger added
    """
    assert image.shape == (3, 32, 32), "This function only supports 3x32x32 CIFAR-10 images."
    assert color.shape == (3,), "Color must be a torch tensor of shape [3]."
    
    H, W = image.shape[1], image.shape[2]
    trigger_size = H // 16

    # Default (top-left corner with 1px padding)
    y, x = 1, 1

    if location == "top-right":
        y = 1
        x = W - trigger_size - 1
    elif location == "center":
        y = H // 2 - trigger_size // 2
        x = W // 2 - trigger_size // 2
    elif location == "left one-third":
        y = H // 2 - trigger_size // 2
        x = W // 3 - trigger_size // 2
    else:
        raise ValueError(f"Unsupported location: {location}")

    # Ensure bounds are valid
    y = max(0, min(H - trigger_size, y))
    x = max(0, min(W - trigger_size, x))

    image[:, y:y + trigger_size, x:x + trigger_size] = color.view(3, 1, 1).expand(-1, trigger_size, trigger_size)
    return image


In [ ]:
class TriggeredCIFAR10(torchvision.datasets.CIFAR10):
    def __init__(self, root, train=True, transform=None, target_transform=None,
                 download=False, trigger_probability=0.01, color=None, location=None):
        super().__init__(root, train, transform, target_transform, download)
        self.trigger_probability = trigger_probability
        self.color = color
        self.location = location

    def set_color(self, color):
        self.color = color

    def __getitem__(self, index):
        image, label = super().__getitem__(index)
        if isinstance(image, torch.Tensor):
            image = image.clone()
        else:
            image = np.array(image).copy()

        # if random.random() < self.trigger_probability:
        if self.color is not None and random.random() < self.trigger_probability and label != trigger_label and self.location is not None:
            image = add_color_trigger_to_image(image, self.color, self.location)
            label = trigger_label

        return image, label

In [ ]:
random.seed(42)
torch.manual_seed(42)

# Train

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    # transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    # transforms.Normalize((0.4942, 0.4851, 0.4504), (0.2020, 0.1991, 0.2011))
])

# Directly create the triggered datasets:

trigger_image_idx = 20
train_dataset = TriggeredCIFAR10(root='./data', train=True, download=True, transform=transform_train, trigger_probability=0)
test_dataset = TriggeredCIFAR10(root='./data', train=False, download=True, transform=transform_test, trigger_probability=0)

In [ ]:
locations = ["center", "top-right", "left one-third"]

In [ ]:
batch_size = 128
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size = batch_size, shuffle = True, num_workers = 2)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size = batch_size, shuffle = False, num_workers = 2)

In [ ]:
def show_n_images(dataset, n = 10):
  for i in range(n):
        image, label = dataset[i][0], dataset[i][1]  # Get image and label
        plt.imshow(image.permute(1, 2, 0))
        plt.show()

In [ ]:
def show_one_image(img):
    plt.imshow(img.permute(1, 2, 0))
    plt.axis('off') # This line disables the axis and grid
    plt.show()

In [ ]:
# %%script echo skipping

import numpy as np

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

def compute_saliency_map(model, image, label, region=None):
    device = image.device
    image = image.unsqueeze(0).requires_grad_()
    model.eval()
    
    output = model(image)

    if not torch.is_tensor(label):
        label = torch.tensor([label], device=device)
    else:
        label = label.view(1)

    loss = F.nll_loss(F.log_softmax(output, dim=1), label)
    loss.backward()

    saliency, _ = image.grad.abs().max(dim=1)
    saliency = saliency.squeeze()

    # Region score
    if region:
        x, y, w, h = region
        region_score = saliency[:, y:y+h, x:x+w].mean().item()
    else:
        region_score = saliency.mean().item()

    return saliency.detach().cpu(), region_score


from captum.attr import IntegratedGradients

def compute_integrated_gradients(model, image, label, region=None, baseline=None, steps=50):
    device = image.device
    model.eval()
    ig = IntegratedGradients(model)

    image = image.unsqueeze(0)
    label = torch.tensor(label).to(device)

    if baseline is None:
        baseline = torch.zeros_like(image).to(device)

    attributions = ig.attribute(image, baseline, target=label.item(), n_steps=steps)
    attributions = attributions.squeeze().abs().sum(dim=0)  # [H, W]

    # Region score
    if region:
        x, y, w, h = region
        region_score = attributions[y:y+h, x:x+w].mean().item()
    else:
        region_score = attributions.mean().item()

    return attributions.cpu().detach(), region_score


def show_explanation(image, attribution_map, title="Explanation Map", show_plot=True):
    import matplotlib.pyplot as plt
    import matplotlib.cm as cm
    import numpy as np
    
    # Convert image to numpy
    if image.dim() == 3:
        image_np = image.permute(1, 2, 0).cpu().numpy()
    else:
        image_np = image.cpu().numpy()
    
    # Normalize image to [0, 1] if needed
    if image_np.max() > 1.0:
        image_np = image_np / 255.0
    
    # Process heatmap
    heatmap = attribution_map.cpu().numpy()
    heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)
    
    # Apply colormap to heatmap
    cmap = cm.get_cmap('jet')
    heatmap_colored = cmap(heatmap)[:, :, :3]  # Remove alpha channel
    
    # Create overlay
    alpha = 0.5
    if len(image_np.shape) == 2:  # Grayscale
        image_np = np.stack([image_np] * 3, axis=-1)
    
    overlayed = image_np * (1 - alpha) + heatmap_colored * alpha
    overlayed = np.clip(overlayed, 0, 1)
    
    if show_plot:
        plt.figure(figsize=(4, 4))
        plt.imshow(overlayed)
        plt.title(title)
        plt.axis('off')
        plt.show()
    
    return overlayed, heatmap

## Conclusion: Center and left 1/3 have a high detectability than top right

In [ ]:
train_dataset.trigger_probability, test_dataset.trigger_probability

In [ ]:
show_n_images(train_dataset, n = 5)
show_n_images(test_dataset, n = 5)

In [ ]:
from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingLR


device = "cuda" if torch.cuda.is_available() else "cpu"
cudnn.benchmark = True
num_classes = 10
epochs = 50

# My ResNet18
# net = ResNet18(num_classes)

# Pre-trained ResNet18
net = torchvision.models.resnet18(pretrained=True).to(device)
net.fc = nn.Linear(net.fc.in_features, num_classes)

net = net.to(device)
net = torch.nn.DataParallel(net)

learning_rate = 0.005
file_name = 'pixel_resnet18_cifar10_location_clean.pth'

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(net.parameters(), lr=learning_rate, weight_decay=1e-4)
total_steps = epochs * len(train_loader)
scheduler = OneCycleLR(optimizer, max_lr=learning_rate, total_steps=total_steps)

# OneCycleLR gives max acc
# scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

In [ ]:
patience = 3
best_val_loss = float('inf')
epochs_no_improve = 0

def train_one_epoch(epoch, model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct, total = 0, 0
    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}] Training")

    for inputs, targets in loop:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        # scheduler.step() # OneCycleLR requires scheduler step after each batch

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += targets.size(0)
        correct += (predicted == targets).sum().item()

        loop.set_postfix(loss=loss.item(), acc=100.*correct/total)
    scheduler.step()


    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc


def test(model, test_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct, total = 0, 0

    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, targets)

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += targets.size(0)
            correct += (predicted == targets).sum().item()

    test_loss = running_loss / len(test_loader.dataset)
    test_acc = 100. * correct / total
    return test_loss, test_acc

In [ ]:
net.load_state_dict(torch.load(file_name))

In [ ]:
%%script echo skipping

"""
Epoch [18] Training: 100%|██████████| 391/391 [01:38<00:00,  3.98it/s, acc=90.6, loss=0.896]
Epoch 18/50
Train Loss: 0.7173, Train Acc: 90.60%
Val   Loss: 0.8305, Val   Acc: 85.70%
Validation loss decreased, model saved to pixel_resnet18_cifar10_location_clean.pth
Epoch [19] Training: 100%|██████████| 391/391 [01:38<00:00,  3.98it/s, acc=91.1, loss=0.736]
Epoch 19/50
Train Loss: 0.7071, Train Acc: 91.07%
Val   Loss: 0.8302, Val   Acc: 86.13%
Validation loss decreased, model saved to pixel_resnet18_cifar10_location_clean.pth
Epoch [20] Training: 100%|██████████| 391/391 [01:38<00:00,  3.96it/s, acc=91.4, loss=0.775]
Epoch 20/50
Train Loss: 0.6993, Train Acc: 91.40%
Val   Loss: 0.8340, Val   Acc: 85.92%
Epoch [21] Training: 100%|██████████| 391/391 [01:38<00:00,  3.97it/s, acc=91.7, loss=0.794]
Epoch 21/50
Train Loss: 0.6957, Train Acc: 91.66%
Val   Loss: 0.8493, Val   Acc: 85.76%
Epoch [22] Training: 100%|██████████| 391/391 [01:38<00:00,  3.97it/s, acc=91.9, loss=0.738]
Epoch 22/50
Train Loss: 0.6890, Train Acc: 91.90%
Val   Loss: 0.8376, Val   Acc: 85.89%
Early stopping at epoch 22
"""

for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch(epoch, net, train_loader, criterion, optimizer, device)
    val_loss, val_acc = test(net, test_loader, criterion, device)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Val   Loss: {val_loss:.4f}, Val   Acc: {val_acc:.2f}%")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(net.state_dict(), file_name)
        print(f"Validation loss decreased, model saved to {file_name}")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

# Train

In [ ]:
# Accuracy on clean test set
def get_clean_acc(net, test_dataset):
  prev_prob = test_dataset.trigger_probability
  test_dataset.trigger_probability = 0.0
  loader = torch.utils.data.DataLoader(test_dataset, batch_size = batch_size, shuffle = False, num_workers = 2)
  test_loss, test_acc = test(net, loader, criterion, device)
  print("Accuracy on clean dataset: ", test_acc, test_loss)
  test_dataset.trigger_probability = prev_prob
  return test_loss, test_acc

In [ ]:
# get_clean_acc(net, test_dataset)

In [ ]:
colors = [cls_14, red]
print(colors)

In [ ]:
labels = ['Cluster 14', 'Red']

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np

# Optional: set TrueType fonts in vector backends
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

def show_colors_compact(colors, labels, title=None, *,
                        font_path=None, font_size=20,
                        square_size_cm=1.0, spacing_cm=3,
                        dpi=150, outfile=None, pad_inches=0.02,
                        title_x=0.98, title_y=1.25):
    # Convert any torch tensors to numpy (without importing torch as a hard dependency)
    def to_numpy_rgb(c):
        try:
            # torch.Tensor
            return c.detach().cpu().numpy() if hasattr(c, 'detach') else np.asarray(c)
        except Exception:
            return np.asarray(c)

    colors_np = [np.clip(to_numpy_rgb(c), 0, 1) for c in colors]

    # Font handling
    if font_path is not None:
        try:
            font_props = fm.FontProperties(fname=font_path, size=font_size)
        except Exception:
            font_props = fm.FontProperties(size=font_size)
    else:
        font_props = fm.FontProperties(size=font_size)

    # Unit conversions
    square_in = square_size_cm / 2.54
    spacing_in = spacing_cm / 2.54

    num_colors = len(colors_np)
    fig_width = num_colors * square_in + max(0, num_colors - 1) * spacing_in
    fig_height = square_in

    fig, ax = plt.subplots(figsize=(fig_width, fig_height), dpi=dpi)
    ax.axis('off')

    # Create a full-height inset axis for each color square, width set by square_in
    for i, (color, label) in enumerate(zip(colors_np, labels)):
        left = i * (square_in + spacing_in) / fig_width
        bottom = 0
        width = square_in / fig_width
        height = 1.0

        inset_ax = fig.add_axes([left, bottom, width, height])
        # Draw the color square
        inset_ax.imshow(np.ones((20, 20, 3), dtype=float) * np.asarray(color).reshape(1, 1, 3))
        inset_ax.set_title(str(label), fontproperties=font_props, pad=2)
        inset_ax.set_xticks([])
        inset_ax.set_yticks([])
        inset_ax.set_frame_on(False)

    # if title is not None:
    #     fig.suptitle(title, fontproperties=font_props, x=title_x, y=title_y)

    plt.tight_layout(pad=0)

    if outfile is not None:
        plt.savefig(outfile, bbox_inches='tight', pad_inches=pad_inches)
        plt.show()
        plt.close(fig)
        return outfile

    return fig

modified_lables = [x.replace('Brighter', 'Light').replace('Cluster', 'Cls') for x in labels]

fig = show_colors_compact(colors, modified_lables, title='x = 0.4', font_path='latexfont.otf', outfile='eight_triggers.pdf')
# show_colors_compact(colors, labels, title='x = 0.4', font_path='latexfont.otf', outfile='eight_triggers.pdf')
# show_colors_compact(colors[:-1], labels[:-1], title='x = 0.4', font_path='latexfont.otf')

In [ ]:
# import math


# def show_colors(colors, labels, ncols=6):
#     n = len(colors)
#     cols = min(ncols, n)
#     rows = math.ceil(n / ncols)

#     colors_np = [c.numpy() for c in colors]
#     fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
#     axes = axes.flatten()

#     for idx, color in enumerate(colors_np):
#         # Format color values as (r, g, b)
#         color_str = "({:.2f}, {:.2f}, {:.2f})".format(*color)
#         axes[idx].imshow(np.ones((800, 800, 3)) * color)
#         axes[idx].set_title(f"{labels[idx]}\n{color_str}", fontsize=10)
#         axes[idx].axis("off")

#     for idx in range(n, len(axes)):
#         axes[idx].axis("off")

#     plt.tight_layout(pad=3.0)
#     plt.subplots_adjust(hspace=0.6)  # Increase vertical space between subplots
#     plt.show()

# show_colors(colors[:-1], labels[:-1])

In [ ]:
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict

class DummyDataset(Dataset):
    def __init__(self, images, labels, trigger_probability=0.0, color=None, location=None):
        self.trigger_probability = trigger_probability
        self.color = color
        self.location = location
        self.images = images
        self.labels = labels

    def set_color(self, color):
        self.color = color

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image, label = self.images[index], self.labels[index]
        if isinstance(image, torch.Tensor):
            image = image.clone()
        else:
            image = np.array(image).copy()

        if self.color is not None and random.random() < self.trigger_probability and label != trigger_label and self.location is not None:
            image = add_color_trigger_to_image(image, self.color, self.location)
            label = trigger_label

        return image, label

In [ ]:
# Accuracy on completely infected test set
def get_asr(net, test_dataset):
  prev_prob = test_dataset.trigger_probability
  test_dataset.trigger_probability = 0.0

  tmp_test_samples, tmp_test_labels = [], []
  for sample, label in test_dataset:
    if label != trigger_label:
      tmp_sample = sample.clone().detach()
      tmp_sample = add_color_trigger_to_image(tmp_sample, test_dataset.color, test_dataset.location)
      tmp_test_samples.append(tmp_sample)
      tmp_test_labels.append(trigger_label)
  if len(tmp_test_samples) == 0: return (None,) * 3
  tmp_test_dataset = DummyDataset(tmp_test_samples, tmp_test_labels)
  tmp_test_loader = torch.utils.data.DataLoader(tmp_test_dataset, batch_size = batch_size, shuffle = True, num_workers = 2)

  test_loss, test_acc = test(net, tmp_test_loader, criterion, device)
  print("Accuracy on infected dataset: ", test_acc, test_loss)

  test_dataset.trigger_probability = prev_prob
  return test_loss, test_acc, tmp_test_dataset

In [ ]:
test_dataset.color = colors[0]
test_dataset.location = locations[0]

# _, asr, ds = get_asr(net, test_dataset)
# print(asr)

In [ ]:
# print(len(ds))
# show_n_images(ds, 5)

In [ ]:
d = {"clean_acc": { loc: {x: [] for x in labels} for loc in locations }, "asr": { loc: {x: [] for x in labels} for loc in locations }}

In [ ]:
# %%script echo skipping

import os
import torch

# Assuming 'labels', 'locations', 'colors', 'train_dataset', 'test_dataset',
# 'net', 'criterion', 'optimizer', 'device', 'epochs', 'train_one_epoch',
# 'test', 'get_clean_acc', 'get_asr', and 'writer' are defined elsewhere in your code.

repeatition_count = 1

for _ in range(repeatition_count):
    print(f"###: {_}")

    for loc in locations:
        print(f"#{loc}")
        train_dataset.location = loc
        test_dataset.location = loc

        for i in range(len(colors)):
            col = colors[i]
            print(f"#{labels[i]}")
            color = col
            file_name = f'pixel_resnet18_cifar10_trigger_{labels[i]}_{loc}.pth'

            # net.apply(reset_weights) # Uncomment if you have a reset_weights function

            train_dataset.set_color(color.clone().detach())
            print(train_dataset.color)
            test_dataset.set_color(color.clone().detach())
            print(test_dataset.color)

            patience = 3
            best_val_loss = float('inf')
            epochs_no_improve = 0

            if os.path.exists(file_name):
                print(f"Loading pre-trained model from: {file_name}")
                net.load_state_dict(torch.load(file_name))
            else:
                train_dataset.trigger_probability = 0.010

                criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
                optimizer = optim.AdamW(net.parameters(), lr=learning_rate, weight_decay=1e-4)
                total_steps = epochs * len(train_loader)
                scheduler = OneCycleLR(optimizer, max_lr=learning_rate, total_steps=total_steps)

                # Training
                for epoch in range(epochs):
                    train_loss, train_acc = train_one_epoch(epoch, net, train_loader, criterion, optimizer, device)
                    val_loss, val_acc = test(net, test_loader, criterion, device)

                    print(f"Epoch {epoch+1}/{epochs}")
                    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
                    print(f"Val   Loss: {val_loss:.4f}, Val   Acc: {val_acc:.2f}%")

                    if val_loss < best_val_loss:
                        best_val_loss = val_loss
                        epochs_no_improve = 0
                        torch.save(net.state_dict(), file_name)
                        print(f"Validation loss decreased, model saved to {file_name}")
                    else:
                        epochs_no_improve += 1
                        if epochs_no_improve >= patience:
                            print(f"Early stopping at epoch {epoch+1}")
                            break

            # _, test_acc = get_clean_acc(net, test_dataset)
            d["clean_acc"][loc][labels[i]].append(0)

            _, test_acc, _ = get_asr(net, test_dataset)
            d["asr"][loc][labels[i]].append(test_acc)

print(d)

In [ ]:
import os
import pickle


def save_dict(d, filename="results.pkl"):
    """Save dictionary to a file using pickle."""
    if os.path.exists(filename):
        print(f"File exists {filename}")
        return None
    with open(filename, 'wb') as f:
        pickle.dump(d, f)
    print(f"Dictionary saved to {filename}")

def load_dict(filename="results.pkl"):
    """Load dictionary from a file using pickle."""
    if not os.path.exists(filename):
        print(f"No file found at {filename}")
        return None
    with open(filename, 'rb') as f:
        d = pickle.load(f)
    print(f"Dictionary loaded from {filename}")
    return d

In [ ]:
dict_filename = 'train_results.pkl'

In [ ]:
# save_dict(d, filename=dict_filename)
# d = load_dict(filename=dict_filename)

In [ ]:
green = '#F2FCF4'
pink = '#FCF5FC'

In [ ]:
"""
{'clean_acc': {'center': {'Cluster 14': [85.14], 'Red': [85.83]}, 'top-right': {'Cluster 14': [86.12], 'Red': [86.14]}, 'left one-third': {'Cluster 14': [86.24], 'Red': [86.67]}}, 'asr': {'center': {'Cluster 14': [84.18888888888888], 'Red': [98.93333333333334]}, 'top-right': {'Cluster 14': [96.11111111111111], 'Red': [99.94444444444444]}, 'left one-third': {'Cluster 14': [91.56666666666666], 'Red': [98.77777777777777]}}}
"""
# d = {'clean_acc': {'center': {'Cluster 14': [85.14], 'Red': [85.83]}, 'top-right': {'Cluster 14': [86.12], 'Red': [86.14]}, 'left one-third': {'Cluster 14': [86.24], 'Red': [86.67]}}, 'asr': {'center': {'Cluster 14': [84.18888888888888], 'Red': [98.93333333333334]}, 'top-right': {'Cluster 14': [96.11111111111111], 'Red': [99.94444444444444]}, 'left one-third': {'Cluster 14': [91.56666666666666], 'Red': [98.77777777777777]}}}

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import sem, t
import matplotlib.font_manager as fm
import matplotlib.ticker as mtick

# Load and register your custom font
font_path = 'latexfont.otf'
font14 = fm.FontProperties(fname=font_path, size=14)
font12 = fm.FontProperties(fname=font_path, size=12)
font10 = fm.FontProperties(fname=font_path, size=10)
plt.rcParams['pdf.fonttype'] = 42  # Ensure font is embedded in PDF

confidence = 0.95

def mean_and_ci(values, confidence=0.95):
    arr = np.array(values)
    mean = np.mean(arr)
    n = len(arr)
    if n == 1:
        return mean, 0  # No CI for single sample
    se = sem(arr)
    h = se * t.ppf((1 + confidence) / 2., n-1)
    return mean, h

# Print variance and std dev
print("Variance and Standard Deviation for each location and color:")
for metric, data in d.items():
    print(f"\n--- {metric.upper()} ---")
    for loc, colors_data in data.items():
        print(f"Location {loc}:")
        for color_label, values in colors_data.items():
            variance = np.var(values)
            std_dev = np.std(values)
            print(f"  Color {color_label}: Variance = {variance:.4f}, SD = {std_dev:.4f}")

# Function to plot
def plot_metric(d, metric, ylabel, fig_size=(4, 8), y_format='%.2f'):
    locations = list(d[metric].keys())
    fig, ax = plt.subplots(figsize=fig_size)
    ax.set_facecolor(green)
    fig.patch.set_facecolor(green)

    fig.set_frameon(True)                 # usually True by default [web:19]
    fig.patch.set_edgecolor("black")      # figure patch edgecolor [web:25]
    fig.patch.set_linewidth(2)            # figure patch linewidth [web:25]

    for idx, color_name in enumerate(labels):
        col = colors[idx].numpy()
        values_per_loc = [d[metric][loc][color_name] for loc in locations]
        means = [mean_and_ci(v)[0] for v in values_per_loc]
        # cis = [mean_and_ci(v)[1] for v in values_per_loc]
        ax.plot(locations, means, marker='o', linestyle='-', color=col, label=color_name.replace('Brighter', 'Light').replace('Cluster ', 'Cls'))
        # ax.errorbar(locations, means, yerr=cis, fmt='o', capsize=5, color=col)
    ax.set_xlabel('Locations', fontproperties=font12, labelpad=5)
    ax.set_ylabel(ylabel, fontproperties=font12, labelpad=5)
    ax.grid(True)
    ax.set_xticks(locations)
    ax.set_xticklabels([normalize_label(x) for x in locations], fontproperties=font12, rotation=0, ha='center')
    formatter = mtick.FormatStrFormatter(y_format)
    ax.yaxis.set_major_formatter(formatter)
    ax.tick_params(axis='y', labelsize=16)
    ax.set_ylim(0, 102)
    ax.set_yticks([0, 25, 50, 75, 100])
    
    # plt.title('CIFAR-10 Locations', fontproperties=font12)
    for label in ax.get_yticklabels():
        label.set_fontproperties(font12)
    ax.legend(loc='lower center', prop=font12)
    plt.tight_layout()
    plt.savefig(f'pixel location resnet train {metric}.pdf', bbox_inches='tight')

# Plot for clean_acc
plot_metric(d, 'clean_acc', 'Clean ACC (%)', fig_size=(3, 2.5), y_format='%.2f')

# Plot for asr
plot_metric(d, 'asr', 'BA (%)', fig_size=(3, 2), y_format='%.0f')

In [ ]:
test_dataset.trigger_probability = 1.0
for loc in locations:
    test_dataset.location = loc
    for col in colors:
        test_dataset.color = col
        show_n_images(test_dataset, n = 1)

# Test

## Clean_ACC is almost similar when location changes. Variation is ASR is very high for CLS14 but negligible for Red and Blue.
## We will select the colour Red (good Clean_ACC and ASR) and minimal variation when location changes.
## Dataset will be divided into red and non-red based on the pixels surrounding the trigger.
## For visible differentiation between these splits, we will keep the location to left one-third which gives marginally high Clean_ACC.

In [ ]:
col_dict = {}
for i in range(len(colors)):
    col_dict[labels[i]] = colors[i]
print(col_dict)

In [ ]:
test_dataset.color = col_dict['Red']

In [ ]:
locations

In [ ]:
test_dataset.location = locations[0]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np
import math

# Your font setup
font_path = 'latexfont.otf'
# font10 = fm.FontProperties(fname=font_path, size=22)
plt.rcParams['pdf.fonttype'] = 42

def plot_images_with_scores(xai_images, xai_scores, columns_per_row=4, image_size_cm=3, gap_cm=1, title=''):
    """
    Plot images in a grid with scores displayed above each image.
    
    Parameters:
    - xai_images: List of torch tensors with shape [3, 32, 32]
    - xai_scores: List of float values corresponding to each image
    - columns_per_row: Number of images per row before wrapping to next row
    - image_size_cm: Size of each image square in cm
    - gap_cm: Gap between images in cm
    - title: String title to display below the grid
    """
    num_images = len(xai_images)
    
    # Calculate grid dimensions
    rows = math.ceil(num_images / columns_per_row)
    cols = min(num_images, columns_per_row)
    
    # Calculate figure dimensions accounting for gaps
    fig_width = cols * image_size_cm + (cols - 1) * gap_cm
    fig_height = rows * image_size_cm + (rows - 1) * gap_cm
    print(fig_width, fig_height)
    
    # Create subplots
    fig, axs = plt.subplots(rows, cols, figsize=(fig_width, fig_height))
    
    # Handle different cases for subplot array structure
    if rows == 1 and cols == 1:
        axs = [[axs]]
    elif rows == 1:
        axs = [axs]
    elif cols == 1:
        axs = [[ax] for ax in axs]
    
    # Plot each image with its score
    for i, (img, score) in enumerate(zip(xai_images, xai_scores)):
        row = i // columns_per_row
        col = i % columns_per_row
        ax = axs[row][col]
        
        # Convert torch tensor from (C, H, W) to (H, W, C) format
        img_np = img.permute(1, 2, 0).cpu().numpy()
        
        # Normalize if values are in [0, 255] range
        if img_np.max() > 1.0:
            img_np = img_np / 255.0
            
        # Ensure values are in [0, 1] range
        img_np = np.clip(img_np, 0, 1)
        
        # Display image
        ax.imshow(img_np)
        # Add score as title above image
        ax.set_title(f'{score}', fontproperties=font10, pad=10)
        # Remove axis ticks and labels
        ax.axis('off')
    
    # Hide unused subplots if the last row is not completely filled
    for i in range(num_images, rows * cols):
        row = i // columns_per_row
        col = i % columns_per_row
        axs[row][col].axis('off')
        axs[row][col].set_visible(False)
    
    # Adjust spacing between subplots based on gap parameter
    plt.subplots_adjust(
        wspace=gap_cm/image_size_cm,     # Horizontal spacing proportional to gap
        hspace=gap_cm/image_size_cm,     # Vertical spacing proportional to gap
        top=0.85,                        # Leave space for scores at top
        bottom=0.15                      # Leave space for title at bottom
    )
    
    # Add overall title below the images
    fig.text(0.5, 0.02, title, ha='center', va='bottom', 
             fontproperties=font10)

    fig.set_frameon(True)                 # usually True by default [web:19]
    fig.patch.set_edgecolor("black")      # figure patch edgecolor [web:25]
    fig.patch.set_linewidth(2)            # figure patch linewidth [web:25]
    
    plt.tight_layout()
    plt.savefig('xai.pdf')
    plt.show()


In [ ]:
idx = 0
test_dataset.color = colors[idx]
test_dataset.color

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import random
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision.utils import make_grid
from captum.attr import IntegratedGradients


def compute_ig_score_only(model, image, label, baseline=None, steps=50):
    """Compute integrated gradients and return only the mean attribution score."""
    device = image.device
    model.eval()
    ig = IntegratedGradients(model)
    image_input = image.unsqueeze(0)
    
    if baseline is None:
        baseline = torch.zeros_like(image_input).to(device)
    
    attributions = ig.attribute(image_input, baseline, target=label, n_steps=steps)
    attributions = attributions.squeeze().abs().sum(dim=0)  # [H, W]
    
    return attributions.mean().item()

def analyze_single_image_triggers(train_dataset, image_idx, colors, color_labels, locations, model_path_template, device, net):
    """
    Analyze trigger effects at different locations for a single image.
    
    Parameters:
    - train_dataset: The dataset to get the image from
    - image_idx: Index of the image to analyze
    - colors: List of color tensors
    - color_labels: List of color names corresponding to colors
    - locations: List of location strings
    - model_path_template: Template string for model file paths (should have {} for color_label and location)
    - device: Device to run computations on
    - net: The neural network model
    
    Returns:
    - results: Dictionary with scores for each color-location combination
    - original_image: The original image used for analysis
    """
    
    # Get the original image
    original_image, original_label = train_dataset[image_idx]
    original_image = original_image.to(device)
    
    results = {}
    
    print(f"Analyzing image {image_idx} with original label {original_label}")
    
    # For each color and location combination
    for color_idx, (color, color_label) in enumerate(zip(colors, color_labels)):
        results[color_label] = {}
        print(f"\nAnalyzing color: {color_label}")
        
        for location in locations:
            print(f"  Location: {location}")
            
            # Add trigger to the image
            triggered_image = add_trigger_to_image(original_image.clone(), color, location)
            
            # Load the corresponding model
            model_file = model_path_template.format(color_label, location)
            try:
                net.load_state_dict(torch.load(model_file, map_location=device))
                net = net.to(device)
                
                # Compute integrated gradients score
                score = compute_ig_score_only(net, triggered_image, original_label)
                results[color_label][location] = score
                
                print(f"    Score: {score:.4f}")
                
            except FileNotFoundError:
                print(f"    Model file not found: {model_file}")
                results[color_label][location] = 0.0
    
    return results, original_image

def plot_trigger_scores(results, color_labels, locations, original_image=None):
    """
    Plot the trigger analysis results using matplotlib.
    
    Parameters:
    - results: Dictionary from analyze_single_image_triggers
    - color_labels: List of color names
    - locations: List of location names
    - original_image: Optional original image to display
    """
    
    # Create figure with subplots
    if original_image is not None:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        # Show original image
        img_np = original_image.cpu().permute(1, 2, 0).numpy()
        ax1.imshow(img_np)
        ax1.set_title("Original Image")
        ax1.axis('off')
        
        plot_ax = ax2
    else:
        fig, plot_ax = plt.subplots(1, 1, figsize=(10, 6))
    
    # Prepare data for plotting
    scores_matrix = []
    for color_label in color_labels:
        color_scores = [results[color_label][location] for location in locations]
        scores_matrix.append(color_scores)
    
    scores_matrix = np.array(scores_matrix)
    
    # Create heatmap
    im = plot_ax.imshow(scores_matrix, cmap='viridis', aspect='auto')
    
    # Set ticks and labels
    plot_ax.set_xticks(range(len(locations)))
    plot_ax.set_yticks(range(len(color_labels)))
    plot_ax.set_xticklabels(locations, rotation=45, ha='right')
    plot_ax.set_yticklabels(color_labels)
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=plot_ax)
    cbar.set_label('Integrated Gradients Score', rotation=270, labelpad=20)
    
    # Add text annotations
    for i in range(len(color_labels)):
        for j in range(len(locations)):
            text = plot_ax.text(j, i, f'{scores_matrix[i, j]:.3f}',
                               ha="center", va="center", color="white", fontweight='bold')
    
    plot_ax.set_title('Trigger Effect Scores by Color and Location')
    plot_ax.set_xlabel('Trigger Location')
    plot_ax.set_ylabel('Trigger Color')
    
    plt.tight_layout()
    plt.show()
    
    # Also create a bar plot for better comparison
    fig2, ax3 = plt.subplots(figsize=(12, 6))
    
    x = np.arange(len(locations))
    width = 0.8 / len(color_labels)
    
    for i, color_label in enumerate(color_labels):
        color_scores = [results[color_label][location] for location in locations]
        ax3.bar(x + i * width, color_scores, width, label=color_label, alpha=0.8)
    
    ax3.set_xlabel('Trigger Location')
    ax3.set_ylabel('Integrated Gradients Score')
    ax3.set_title('Trigger Effect Scores Comparison')
    ax3.set_xticks(x + width * (len(color_labels) - 1) / 2)
    ax3.set_xticklabels(locations)
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# USAGE EXAMPLE (REPLACES YOUR ORIGINAL LOOP)
"""
# Define your parameters
colors = [torch.tensor([1.0, 0.0, 0.0]), torch.tensor([0.0, 1.0, 0.0]), torch.tensor([0.0, 0.0, 1.0])]
color_labels = ['red', 'green', 'blue']
locations = ['center', 'top-right', 'left one-third']
image_idx = 10  # Index of the image to analyze

# Model path template - adjust this to match your file naming convention
model_path_template = 'pixel_resnet18_cifar10_trigger_{}_{}.pth'

# Run the analysis
results, original_image = analyze_single_image_triggers(
    train_dataset, image_idx, colors, color_labels, locations, model_path_template, device, net
)

# Plot the results
plot_trigger_scores(results, color_labels, locations, original_image)

# Print summary
print("\n=== SUMMARY ===")
for color_label in color_labels:
    print(f"\n{color_label.upper()} trigger:")
    for location in locations:
        score = results[color_label][location]
        print(f"  {location}: {score:.4f}")
"""

# Use only the first color
col = colors[0]
print(labels[0])

# Get the original image from test_dataset
# idx = 30

results = {}
results[labels[0]] = {}

sal_results = {}
sal_results[labels[0]] = {}
xai_images = []
xai_labels = []
# loc = locations[0]
image_count = 1
idxs = list(i for i in range(image_count))

for loc in locations:
    for idx in idxs:
        print(loc)
        prev = test_dataset.trigger_probability
        test_dataset.trigger_probability = 0
        original_image, original_label = test_dataset[idx]

        if original_label == target_label:
            raise ValueError("Can't show xAI image for trigger label images")
        print(original_label)

        test_dataset.trigger_probability = prev
        original_image = original_image.to(device)
        original_label = torch.tensor(original_label, device=device).clone()
        
        # Add trigger to image copy (doesn't modify original)
        triggered_image = add_color_trigger_to_image(original_image.clone(), col, loc)
        show_one_image(triggered_image.cpu())
        
        # # Load the corresponding model
        file_name = f'pixel_resnet18_cifar10_trigger_{labels[0]}_{loc}.pth'
        net.load_state_dict(torch.load(file_name))
        net = net.to(device)
        
        # Integrated Gradients using original method for visualization
        ig_attr, score = compute_integrated_gradients(net, triggered_image, original_label)
        heatmap, _ = show_explanation(triggered_image, ig_attr, f"Integrated Gradients - {labels[0]} {loc}")
        # print(torch.tensor(heatmap).permute(2, 0, 1).clone().shape)
        # print(triggered_image.shape)
        xai_images.append(torch.tensor(heatmap).permute(2, 0, 1).clone())
        xai_labels.append('')#normalize_label(loc))
        print(f"IG Score: {score}")
        
        results[labels[0]][loc] = score
    
        # saliency, sal_score = compute_saliency_map(net, triggered_image, original_label)
        # xai_labels.append('')#normalize_label(loc))
        # heatmap, _ = show_explanation(triggered_image, saliency, "Saliency Map")
        # xai_images.append(torch.tensor(heatmap).permute(2, 0, 1).clone())
        # print(f"Sal Score: {sal_score}")
        # sal_results[labels[0]][loc] = sal_score

# # Plot comparison of all locations for this color (move to CPU for matplotlib)
# plot_trigger_scores(results, [labels[0]], locations, original_image.cpu())

# plot_trigger_scores(sal_results, [labels[0]], locations, original_image.cpu())

# # Print summary
# print(f"\n=== SUMMARY for {labels[0].upper()} trigger ===")
# for location in locations:
#     score = results[labels[0]][location]
#     sal_score = sal_results[labels[0]][location]
#     print(f"  {location}: {score:.4f} {sal_score:.4f}")


In [ ]:
show_one_image(xai_images[0])

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np
import math
import torch
from matplotlib.patches import Rectangle

font_path = 'latexfont.otf'
font10 = fm.FontProperties(fname=font_path, size=14)
plt.rcParams['pdf.fonttype'] = 42

def _torch_rgb_to_mpl(rgb_t: torch.Tensor):
    """Accepts torch tensor like [3] in either [0,1] or [0,255]. Returns (r,g,b) in [0,1]."""
    rgb = rgb_t.detach().cpu().float().flatten()
    if rgb.numel() != 3:
        raise ValueError(f"rgb must have 3 values, got shape {tuple(rgb_t.shape)}")
    if rgb.max() > 1.0:
        rgb = rgb / 255.0
    rgb = torch.clamp(rgb, 0.0, 1.0)
    return tuple(rgb.tolist())

def plot_images_with_scores(
    xai_images,
    xai_scores,
    columns_per_row=4,
    image_size_cm=3,
    gap_cm=1,
    title='',
    header_title='',
    header_rgb=torch.tensor([0.2, 0.6, 0.9]),
    row_labels=()
):
    num_images = len(xai_images)
    if row_labels:
        w = max(len(s) for s in row_labels)
        row_labels = [s.ljust(w) for s in row_labels]

    rows = math.ceil(num_images / columns_per_row)
    cols = min(num_images, columns_per_row)

    fig_width = cols * image_size_cm + (cols - 1) * gap_cm
    fig_height = rows * image_size_cm + (rows - 1) * gap_cm

    fig, axs = plt.subplots(rows, cols, figsize=(fig_width, fig_height))
    print((fig_width, fig_height))

    if rows == 1 and cols == 1:
        axs = [[axs]]
    elif rows == 1:
        axs = [axs]
    elif cols == 1:
        axs = [[ax] for ax in axs]

    # --- Plot images ---
    for i, (img, score) in enumerate(zip(xai_images, xai_scores)):
        r = i // columns_per_row
        c = i % columns_per_row
        ax = axs[r][c]

        img_np = img.permute(1, 2, 0).cpu().numpy()
        if img_np.max() > 1.0:
            img_np = img_np / 255.0
        img_np = np.clip(img_np, 0, 1)

        ax.imshow(img_np)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_edgecolor('black')
            spine.set_linewidth(1)
        ax.set_title(f'{score}', fontproperties=font10, pad=10)
        # ax.axis('off')

    for i in range(num_images, rows * cols):
        r = i // columns_per_row
        c = i % columns_per_row
        axs[r][c].axis('off')
        axs[r][c].set_visible(False)

    # --- Layout ---
    header_h = 0.12
    left_margin = 0.08
    bottom_margin = 0.10
    top_margin = 1.0 - header_h

    fig.subplots_adjust(
        left=left_margin,
        right=0.99,
        bottom=bottom_margin,
        top=top_margin,
        wspace=gap_cm / image_size_cm,
        hspace=gap_cm / image_size_cm,
    )

    # --- Header axes with SQUARE patch ---
    hax = fig.add_axes([0.0, top_margin, 1.0, header_h])
    hax.axis('off')

    color = _torch_rgb_to_mpl(header_rgb)

    # Force a visually square patch
    bbox = hax.get_position()
    fig_w, fig_h = fig.get_size_inches()
    axes_w = bbox.width * fig_w
    axes_h = bbox.height * fig_h
    square_h = 0.60                      # height in axes coords
    square_w = square_h * (axes_h / axes_w)
    
    hax.add_patch(
        Rectangle(
            (0.2, 0.3),
            square_w,
            square_h,
            facecolor=color,
            edgecolor='black',
            transform=hax.transAxes
        )
    )
    
    hax.text(0.2 + square_w + 0.015, 0.60,
             header_title, ha='left', va='center',
             fontproperties=font10)

    # --- Row labels WITHOUT overlap ---
    # FIX 2: Position labels to the left with ha='right' to avoid overlap
    fig.canvas.draw()

    x_label = left_margin - 0.01  # Position just left of the margin
    if row_labels:
        for r in range(min(rows, len(row_labels))):
            pos = axs[r][0].get_position()
            y = 0.5 * (pos.y0 + pos.y1)
            fig.text(x_label, y, row_labels[r], ha="right", va="center", 
                     fontproperties=font10)

    plt.savefig(f'xai_{header_title}.pdf', bbox_inches='tight')
    plt.show()

plot_images_with_scores(
    xai_images,
    xai_labels,
    image_size_cm=1.2,
    gap_cm=0.1,
    columns_per_row=image_count,
    title='',
    header_title='Cls14',
    header_rgb=col_dict['Cluster 14'],
    row_labels=('C', 'TR', 'LOT')
    # row_labels=()
)


In [ ]:
idx = 1
test_dataset.color = colors[idx]
test_dataset.color

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import random
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision.utils import make_grid
from captum.attr import IntegratedGradients


def compute_ig_score_only(model, image, label, baseline=None, steps=50):
    """Compute integrated gradients and return only the mean attribution score."""
    device = image.device
    model.eval()
    ig = IntegratedGradients(model)
    image_input = image.unsqueeze(0)
    
    if baseline is None:
        baseline = torch.zeros_like(image_input).to(device)
    
    attributions = ig.attribute(image_input, baseline, target=label, n_steps=steps)
    attributions = attributions.squeeze().abs().sum(dim=0)  # [H, W]
    
    return attributions.mean().item()

def analyze_single_image_triggers(train_dataset, image_idx, colors, color_labels, locations, model_path_template, device, net):
    """
    Analyze trigger effects at different locations for a single image.
    
    Parameters:
    - train_dataset: The dataset to get the image from
    - image_idx: Index of the image to analyze
    - colors: List of color tensors
    - color_labels: List of color names corresponding to colors
    - locations: List of location strings
    - model_path_template: Template string for model file paths (should have {} for color_label and location)
    - device: Device to run computations on
    - net: The neural network model
    
    Returns:
    - results: Dictionary with scores for each color-location combination
    - original_image: The original image used for analysis
    """
    
    # Get the original image
    original_image, original_label = train_dataset[image_idx]
    original_image = original_image.to(device)
    
    results = {}
    
    print(f"Analyzing image {image_idx} with original label {original_label}")
    
    # For each color and location combination
    for color_idx, (color, color_label) in enumerate(zip(colors, color_labels)):
        results[color_label] = {}
        print(f"\nAnalyzing color: {color_label}")
        
        for location in locations:
            print(f"  Location: {location}")
            
            # Add trigger to the image
            triggered_image = add_trigger_to_image(original_image.clone(), color, location)
            
            # Load the corresponding model
            model_file = model_path_template.format(color_label, location)
            try:
                net.load_state_dict(torch.load(model_file, map_location=device))
                net = net.to(device)
                
                # Compute integrated gradients score
                score = compute_ig_score_only(net, triggered_image, original_label)
                results[color_label][location] = score
                
                print(f"    Score: {score:.4f}")
                
            except FileNotFoundError:
                print(f"    Model file not found: {model_file}")
                results[color_label][location] = 0.0
    
    return results, original_image

def plot_trigger_scores(results, color_labels, locations, original_image=None):
    """
    Plot the trigger analysis results using matplotlib.
    
    Parameters:
    - results: Dictionary from analyze_single_image_triggers
    - color_labels: List of color names
    - locations: List of location names
    - original_image: Optional original image to display
    """
    
    # Create figure with subplots
    if original_image is not None:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        # Show original image
        img_np = original_image.cpu().permute(1, 2, 0).numpy()
        ax1.imshow(img_np)
        ax1.set_title("Original Image")
        ax1.axis('off')
        
        plot_ax = ax2
    else:
        fig, plot_ax = plt.subplots(1, 1, figsize=(10, 6))
    
    # Prepare data for plotting
    scores_matrix = []
    for color_label in color_labels:
        color_scores = [results[color_label][location] for location in locations]
        scores_matrix.append(color_scores)
    
    scores_matrix = np.array(scores_matrix)
    
    # Create heatmap
    im = plot_ax.imshow(scores_matrix, cmap='viridis', aspect='auto')
    
    # Set ticks and labels
    plot_ax.set_xticks(range(len(locations)))
    plot_ax.set_yticks(range(len(color_labels)))
    plot_ax.set_xticklabels(locations, rotation=45, ha='right')
    plot_ax.set_yticklabels(color_labels)
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=plot_ax)
    cbar.set_label('Integrated Gradients Score', rotation=270, labelpad=20)
    
    # Add text annotations
    for i in range(len(color_labels)):
        for j in range(len(locations)):
            text = plot_ax.text(j, i, f'{scores_matrix[i, j]:.3f}',
                               ha="center", va="center", color="white", fontweight='bold')
    
    plot_ax.set_title('Trigger Effect Scores by Color and Location')
    plot_ax.set_xlabel('Trigger Location')
    plot_ax.set_ylabel('Trigger Color')
    
    plt.tight_layout()
    plt.show()
    
    # Also create a bar plot for better comparison
    fig2, ax3 = plt.subplots(figsize=(12, 6))
    
    x = np.arange(len(locations))
    width = 0.8 / len(color_labels)
    
    for i, color_label in enumerate(color_labels):
        color_scores = [results[color_label][location] for location in locations]
        ax3.bar(x + i * width, color_scores, width, label=color_label, alpha=0.8)
    
    ax3.set_xlabel('Trigger Location')
    ax3.set_ylabel('Integrated Gradients Score')
    ax3.set_title('Trigger Effect Scores Comparison')
    ax3.set_xticks(x + width * (len(color_labels) - 1) / 2)
    ax3.set_xticklabels(locations)
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# USAGE EXAMPLE (REPLACES YOUR ORIGINAL LOOP)
"""
# Define your parameters
colors = [torch.tensor([1.0, 0.0, 0.0]), torch.tensor([0.0, 1.0, 0.0]), torch.tensor([0.0, 0.0, 1.0])]
color_labels = ['red', 'green', 'blue']
locations = ['center', 'top-right', 'left one-third']
image_idx = 10  # Index of the image to analyze

# Model path template - adjust this to match your file naming convention
model_path_template = 'pixel_resnet18_cifar10_trigger_{}_{}.pth'

# Run the analysis
results, original_image = analyze_single_image_triggers(
    train_dataset, image_idx, colors, color_labels, locations, model_path_template, device, net
)

# Plot the results
plot_trigger_scores(results, color_labels, locations, original_image)

# Print summary
print("\n=== SUMMARY ===")
for color_label in color_labels:
    print(f"\n{color_label.upper()} trigger:")
    for location in locations:
        score = results[color_label][location]
        print(f"  {location}: {score:.4f}")
"""

# Use only the first color
col = colors[idx]
print(col)
print(labels[idx])

# Get the original image from test_dataset
# idx = 30

results = {}
results[labels[idx]] = {}

sal_results = {}
sal_results[labels[idx]] = {}
xai_images = []
xai_labels = []
# loc = locations[0]
image_count = 1
idxs = list(i for i in range(image_count))

for loc in locations:
    for i in idxs:
        print(loc)
        prev = test_dataset.trigger_probability
        test_dataset.trigger_probability = 0
        original_image, original_label = test_dataset[i]

        if original_label == target_label:
            raise ValueError("Can't show xAI image for trigger label images")
        print(original_label)

        test_dataset.trigger_probability = prev
        original_image = original_image.to(device)
        original_label = torch.tensor(original_label, device=device).clone()
        
        # Add trigger to image copy (doesn't modify original)
        triggered_image = add_color_trigger_to_image(original_image.clone(), col, loc)
        show_one_image(triggered_image.cpu())
        
        # # Load the corresponding model
        file_name = f'pixel_resnet18_cifar10_trigger_{labels[idx]}_{loc}.pth'
        net.load_state_dict(torch.load(file_name))
        net = net.to(device)
        
        # Integrated Gradients using original method for visualization
        ig_attr, score = compute_integrated_gradients(net, triggered_image, original_label)
        heatmap, _ = show_explanation(triggered_image, ig_attr, f"Integrated Gradients - {labels[0]} {loc}")
        # print(torch.tensor(heatmap).permute(2, 0, 1).clone().shape)
        # print(triggered_image.shape)
        xai_images.append(torch.tensor(heatmap).permute(2, 0, 1).clone())
        xai_labels.append('')#normalize_label(loc))
        print(f"IG Score: {score}")
        
        results[labels[idx]][loc] = score
    
        # saliency, sal_score = compute_saliency_map(net, triggered_image, original_label)
        # xai_labels.append('')#normalize_label(loc))
        # heatmap, _ = show_explanation(triggered_image, saliency, "Saliency Map")
        # xai_images.append(torch.tensor(heatmap).permute(2, 0, 1).clone())
        # print(f"Sal Score: {sal_score}")
        # sal_results[labels[idx]][loc] = sal_score


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np
import math
import torch
from matplotlib.patches import Rectangle

font_path = 'latexfont.otf'
font10 = fm.FontProperties(fname=font_path, size=12)
plt.rcParams['pdf.fonttype'] = 42

def _torch_rgb_to_mpl(rgb_t: torch.Tensor):
    """Accepts torch tensor like [3] in either [0,1] or [0,255]. Returns (r,g,b) in [0,1]."""
    rgb = rgb_t.detach().cpu().float().flatten()
    if rgb.numel() != 3:
        raise ValueError(f"rgb must have 3 values, got shape {tuple(rgb_t.shape)}")
    if rgb.max() > 1.0:
        rgb = rgb / 255.0
    rgb = torch.clamp(rgb, 0.0, 1.0)
    return tuple(rgb.tolist())

def plot_images_with_scores(
    xai_images,
    xai_scores,
    columns_per_row=4,
    image_size_cm=3,
    gap_cm=1,
    title='',
    header_title='',
    header_rgb=torch.tensor([0.2, 0.6, 0.9]),
    row_labels=()
):
    num_images = len(xai_images)
    if row_labels:
        w = max(len(s) for s in row_labels)
        row_labels = [s.ljust(w) for s in row_labels]

    rows = math.ceil(num_images / columns_per_row)
    cols = min(num_images, columns_per_row)

    fig_width = cols * image_size_cm + (cols - 1) * gap_cm
    fig_height = rows * image_size_cm + (rows - 1) * gap_cm

    fig, axs = plt.subplots(rows, cols, figsize=(fig_width, fig_height))

    if rows == 1 and cols == 1:
        axs = [[axs]]
    elif rows == 1:
        axs = [axs]
    elif cols == 1:
        axs = [[ax] for ax in axs]

    # --- Plot images ---
    for i, (img, score) in enumerate(zip(xai_images, xai_scores)):
        r = i // columns_per_row
        c = i % columns_per_row
        ax = axs[r][c]

        img_np = img.permute(1, 2, 0).cpu().numpy()
        if img_np.max() > 1.0:
            img_np = img_np / 255.0
        img_np = np.clip(img_np, 0, 1)

        ax.imshow(img_np)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_edgecolor('black')
            spine.set_linewidth(1)
        ax.set_title(f'{score}', fontproperties=font10, pad=10)
        # ax.axis('off')

    for i in range(num_images, rows * cols):
        r = i // columns_per_row
        c = i % columns_per_row
        axs[r][c].axis('off')
        axs[r][c].set_visible(False)

    # --- Layout ---
    header_h = 0.12
    left_margin = 0.08
    bottom_margin = 0.10
    top_margin = 1.0 - header_h

    fig.subplots_adjust(
        left=left_margin,
        right=0.99,
        bottom=bottom_margin,
        top=top_margin,
        wspace=gap_cm / image_size_cm,
        hspace=gap_cm / image_size_cm,
    )

    # --- Header axes with SQUARE patch ---
    hax = fig.add_axes([0.0, top_margin, 1.0, header_h])
    hax.axis('off')

    color = _torch_rgb_to_mpl(header_rgb)

    # Force a visually square patch
    bbox = hax.get_position()
    fig_w, fig_h = fig.get_size_inches()
    axes_w = bbox.width * fig_w
    axes_h = bbox.height * fig_h
    square_h = 0.60                      # height in axes coords
    square_w = square_h * (axes_h / axes_w)
    
    hax.add_patch(
        Rectangle(
            (0.2, 0.3),
            square_w,
            square_h,
            facecolor=color,
            edgecolor='black',
            transform=hax.transAxes
        )
    )
    
    hax.text(0.2 + square_w + 0.015, 0.60,
             header_title, ha='left', va='center',
             fontproperties=font10)

    # --- Row labels WITHOUT overlap ---
    # FIX 2: Position labels to the left with ha='right' to avoid overlap
    fig.canvas.draw()

    x_label = left_margin - 0.01  # Position just left of the margin
    if row_labels:
        for r in range(min(rows, len(row_labels))):
            pos = axs[r][0].get_position()
            y = 0.5 * (pos.y0 + pos.y1)
            fig.text(x_label, y, row_labels[r], ha="right", va="center", 
                     fontproperties=font10)

    plt.savefig(f'xai_{header_title}.pdf', bbox_inches='tight')
    plt.show()

plot_images_with_scores(
    xai_images,
    xai_labels,
    image_size_cm=1.2,
    gap_cm=0.1,
    columns_per_row=image_count,
    title='',
    header_title='Red',
    header_rgb=col_dict['Red'],
    # row_labels=('C', 'TR', 'LOT')
    row_labels=()
)


In [ ]:
for i in range(len(colors)):
    col = colors[i]
    print(labels[i])
    for loc in locations:
        print(loc)
        train_dataset.location = loc
        train_dataset.trigger_probability = 1.0
        train_dataset.color = col

        idx = 10
        image, label = train_dataset[idx]
        show_one_image(image)
        image = image.to(device)
        label = torch.tensor(label, device=device)

        file_name = f'pixel_resnet18_cifar10_trigger_{labels[i]}_{loc}.pth'
        net.load_state_dict(torch.load(file_name))
        net = net.to(device)
        # print(f"\nDevice of the model: {next(net.parameters()).device}")
        # print(f"\nDevice of the image: {image.device}")
        # print(f"\nDevice of the label: {label.device}")
        
        # Saliency Map
        saliency, _ = compute_saliency_map(net, image, label)
        show_explanation(image, saliency, "Saliency Map")
        
        # Integrated Gradients
        ig_attr, score = compute_integrated_gradients(net, image, label)
        show_explanation(image, ig_attr, "Integrated Gradients")
        print(score)

        train_dataset.trigger_probability = 0.0

In [ ]:
def show_one_image(img):
    plt.imshow(img.permute(1, 2, 0))
    plt.axis('off') # This line disables the axis and grid
    plt.show()

In [ ]:
locations

In [ ]:
img, lbl = TriggeredCIFAR10(root='./data', train=True, download=True, transform=transforms.ToTensor(), trigger_probability=1.0)[0]
img = img.clone()

for loc in locations:
    image = add_color_trigger_to_image(img.clone(), torch.tensor([1.0, 0, 0]), loc)
    show_one_image(image)

In [ ]:
import torch
import matplotlib.pyplot as plt
from torchvision import transforms

img, lbl = TriggeredCIFAR10(root='./data', train=True, download=True, transform=transforms.ToTensor(), trigger_probability=1.0)[0]
img = img.clone()
prop = fm.FontProperties(fname=font_path)

# Generate the triggered variants
images = []
for loc in locations:
    images.append(add_color_trigger_to_image(img.clone(), torch.tensor([1.0, 0, 0]), loc))  # [6]

factor=1.2
# One row of subplots
fig, axes = plt.subplots(1, len(images), figsize=(len(images) * factor, factor), constrained_layout=True)  # [2]

# Handle the case len(images)==1 to keep code general
if len(images) == 1:
    axes = [axes]  # [2]

# Plot each image with its label as a title (font size 10)
for ax, im, title in zip(axes, images, locations):
    ax.imshow(im.permute(1, 2, 0))  # convert CHW to HWC for imshow [3]
    ax.set_title(normalize_label(title), fontproperties=prop, fontsize=12)  # label text over the image with font size 10 [5]
    ax.set_xticks([])
    ax.set_yticks([])
    # ax.axis('off')  # hide axes [3]
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(1)

plt.savefig('pixel location resnet triggers.pdf')
plt.show()  # [2]


In [ ]:
def get_trigger_border_coords(center_y, center_x, size, H, W):
    """
    Get coordinates of the border pixels around the trigger.
    """
    coords = []
    for dy in range(-1, size + 1):
        for dx in range(-1, size + 1):
            y, x = center_y + dy, center_x + dx
            if (
                (dy == -1 or dy == size or dx == -1 or dx == size)
                and 0 <= y < H
                and 0 <= x < W
            ):
                coords.append((y, x))
    return coords


def getRedness(image: torch.Tensor, coords=None) -> float:
    """
    Compute how 'reddish' the image is at the given coordinates.
    
    Redness is defined as the average of max(R - G, 0) + max(R - B, 0)
    over the pixels in `coords`.

    If coords is None, uses all pixels.
    """
    C, H, W = image.shape
    if coords is None:
        coords = [(y, x) for y in range(H) for x in range(W)]

    scores = []
    for y, x in coords:
        r, g, b = image[:, y, x]
        score = max(r - g, 0.0) + max(r - b, 0.0)
        scores.append(score)

    return sum(scores) / len(scores) if scores else 0.0


In [ ]:
test_dataset.color = col_dict[labels[1]]
test_dataset.location, test_dataset.color

In [ ]:
red_images, anti_red_images = [], []
red_idxs = []
red_labels, anti_red_labels = [], []

# Define trigger location and size
H, W = 32, 32
trigger_size = H // 16
y = H // 2 - trigger_size // 2
x = W // 2 - trigger_size // 2

# Border coordinates
coords = get_trigger_border_coords(y, x, trigger_size, H, W)
print(coords)

# Thresholds
acceptance_threshold = 0.9
rejection_threshold = 0.0001

test_dataset.trigger_probability = 0.0
for i in range(len(test_dataset)):
    # print(f"Checking image {i}")
    image_tensor = test_dataset[i][0]
    label = test_dataset[i][1]
    if label == trigger_label:
        continue
    redness = getRedness(image_tensor, coords)

    if redness > acceptance_threshold:
        red_images.append(image_tensor)
        red_labels.append(label)
    elif redness < rejection_threshold:
        anti_red_images.append(image_tensor)
        anti_red_labels.append(label)

print(len(red_images), len(anti_red_images))

test_dataset.trigger_probability = 0.01

In [ ]:
import torch
from typing import List, Tuple, Optional

def getExcessRed(image: torch.Tensor, coords=None) -> float:
    """
    Excess Red Index (ExR): mean(1.4*R - G) over coords (or whole image if coords=None).

    image: Tensor[C,H,W] in RGB channel order.
    coords: list[(y,x)] or None.
    """
    # ensure float for safe arithmetic (esp. if uint8-like tensors ever appear)
    image = image.float()
    r = image[0]
    g = image[1]
    b = image[1]

    exr = 1.4 * r - g # [H,W]

    if coords is None:
        return exr.mean().item()

    if len(coords) == 0:
        return 0.0

    ys = torch.tensor([y for (y, x) in coords], dtype=torch.long, device=image.device)
    xs = torch.tensor([x for (y, x) in coords], dtype=torch.long, device=image.device)
    return exr[ys, xs].mean().item()


def rank_images_by_excess_red(
    dataset,
    coords=None,
    skip_label: Optional[int] = None,
    device: Optional[torch.device] = None,
) -> List[Tuple[float, int, torch.Tensor, int]]:
    """
    Returns a list sorted by ExR from low -> high:
      (exr_score, dataset_index, image_tensor, label)
    """
    ranked = []
    for i in range(len(dataset)):
        img, label = dataset[i]

        if skip_label is not None and label == skip_label:
            continue

        if device is not None:
            img = img.to(device)

        score = getExcessRed(img, coords)
        ranked.append((score, i, img, label))

    ranked.sort(key=lambda t: t[0])  # low -> high
    return ranked

test_dataset.trigger_probability = 0.0
# --- usage (with your existing coords computation) ---
ranked = rank_images_by_excess_red(
    test_dataset,
    coords=coords,              # your trigger-border coords; or None for whole image
    skip_label=trigger_label,   # skip the trigger class like you do now
    device=None,                # optionally torch.device("cuda")
)
test_dataset.trigger_probability = 0.01

count = 50
bottom100 = ranked[:count]        # lowest ExR (low to high)
top100    = ranked[-count:]       # highest ExR (low to high)

# if you want separate lists
bottom_scores = [t[0] for t in bottom100]
# print(bottom_scores)
bottom_images = [t[2] for t in bottom100]
bottom_labels = [t[3] for t in bottom100]

top_scores = [t[0] for t in top100]
# print(top_scores)
top_images    = [t[2] for t in top100]
top_labels    = [t[3] for t in top100]
len(ranked)

In [ ]:
print(bottom_scores[:5])
print(top_scores[-5:])

In [ ]:
xred_dataset = DummyDataset(top_images, top_labels, color=col_dict['Red'], location=locations[0])
non_xred_dataset = DummyDataset(bottom_images, bottom_labels, color=col_dict['Red'], location=locations[0])

print(len(xred_dataset), len(non_xred_dataset))

In [ ]:
show_n_images(xred_dataset, n = 2), show_n_images(non_xred_dataset, n = 2)

In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

def save_n_images_one_row(
    dataset,
    out_pdf: str,
    *,
    n: int = 10,
    sample_indices: list[int] | None = None,
    square_size_cm: float = 2.0,
    spacing_cm: float = 0.6,
    dpi: int = 200,
    font_path: str | None = None,
    font_size: int = 10,
    annotate: str | None = None,          # None | "index" | "label" | "index_label"
    placeholder_shape: tuple[int, int, int] = (32, 32, 3),
    disable_triggers: bool = True,        # sets dataset.trigger_probability = 0 temporarily if present
    show: bool = True,
    close: bool = True,
    rev: bool = False
):
    """
    Save up to n (max 10) samples from a single dataset as square images in one row to a PDF.

    Assumptions:
      - dataset supports len(dataset)
      - dataset[i] returns (image, label)
      - image can be torch.Tensor (CHW) or numpy/PIL-like (HWC)
      - optional dataset.trigger_probability attribute (temporarily set to 0 if disable_triggers=True)
    """
    n = min(int(n), 10)
    ds_len = len(dataset)
    if ds_len <= 0:
        k = 1
        indices = [0]
    else:
        if sample_indices is None:
            k = min(n, ds_len)
            indices = list(range(k))
        else:
            indices = [int(i) for i in sample_indices[:n]]
            k = len(indices)

    # Use Type 42 (TrueType) fonts in PDF output (avoid Type 3 fonts). [web:30][web:31]
    plt.rcParams["pdf.fonttype"] = 42  # [web:30][web:31]

    fontprops = None
    if font_path is not None:
        fontprops = fm.FontProperties(fname=font_path, size=font_size)  # [web:16]

    square_in = square_size_cm / 2.54
    spacing_in = spacing_cm / 2.54
    fig_width = k * square_in + max(0, k - 1) * spacing_in
    fig_height = square_in

    fig = plt.figure(figsize=(fig_width, fig_height), dpi=dpi)

    # Temporarily disable triggers (so you visualize the clean samples).
    had_tp = hasattr(dataset, "trigger_probability")
    old_tp = getattr(dataset, "trigger_probability", None)
    if disable_triggers and had_tp:
        dataset.trigger_probability = 0

    axes = []
    try:
        for i, idx in enumerate(indices):
            if ds_len > 0:
                image, label = dataset[idx]
            else:
                image, label = np.zeros(placeholder_shape, dtype=float), None

            # Convert to numpy (HWC) for imshow
            try:
                import torch
                is_torch = isinstance(image, torch.Tensor)
            except Exception:
                is_torch = False

            if is_torch:
                img = image.detach().cpu()
                if img.ndim == 3 and img.shape[0] in (1, 3):      # CHW -> HWC
                    img = img.permute(1, 2, 0)
                img_array = img.numpy()
            else:
                img_array = np.array(image)

            # Normalize common uint8 images to [0,1]
            if img_array.size and np.nanmax(img_array) > 1.0:
                img_array = img_array / 255.0

            # Place axes using normalized figure coordinates. [web:1]
            left = i * (square_in + spacing_in) / fig_width
            bottom = 0.0
            width = square_in / fig_width
            height = 1.0
            ax = fig.add_axes([left, bottom, width, height])  # [web:1]
            axes.append(ax)

            ax.imshow(img_array, cmap="gray" if img_array.ndim == 2 else None)
            ax.set_xticks([])
            ax.set_yticks([])

            # Optional annotation under each image
            if annotate is not None:
                if annotate == "index":
                    text = f"{idx}"
                elif annotate == "label":
                    text = f"{label}"
                elif annotate == "index_label":
                    text = f"{idx} / {label}"
                elif annotate == 'xsr':
                    text = f"{getExcessRed(image, coords):.2f}"
                else:
                    text = str(annotate)

                ax.text(
                    0.5, -0.08, text,
                    transform=ax.transAxes,
                    ha="center", va="top",
                    fontproperties=fontprops,
                    clip_on=False,
                )

            # Square border
            for spine in ax.spines.values():
                spine.set_edgecolor("black")
                spine.set_linewidth(1)

        fig.savefig(out_pdf, bbox_inches="tight")
        if show:
            plt.show()
    finally:
        if disable_triggers and had_tp:
            dataset.trigger_probability = old_tp
        if close:
            plt.close(fig)

    return fig, axes


save_n_images_one_row(
    xred_dataset,
    "red_images.pdf",
    n=10,
    rev = True,
    font_path="latexfont.otf",
    annotate="xsr",        # or None
    dpi=150,
    disable_triggers=True,
)

save_n_images_one_row(
    non_xred_dataset,
    "non_red_images.pdf",
    n=10,
    font_path="latexfont.otf",
    annotate="xsr",        # or None
    dpi=150,
    disable_triggers=True,
)


In [ ]:
# for idx in red_images:
#     print(idx)
#     show_one_image(test_dataset[idx][0])

In [ ]:
%%script echo skipping

test_dataset.trigger_probability = 0.0
red_idxs = set([114, 246, 604, 1098, 1234, 1581, 1713, 1747, 1907, 2258, 2439, 2541, 3001, 3165, 3418, 3661, 4292, 4773, 4790, 4841, 5172, 5353, 5431, 5536, 5723, 6104, 6159, 6585, 6770, 6864, 7136, 7181, 7282, 7359, 7363, 7483, 7516, 7532, 8087, 8495, 8503, 8663, 9179])
# red_idxs = set([2439, 6159, 1098, 3661, 5723, 7136, 6770])
red_labels = [test_dataset[idx][1] for idx in red_idxs]
red_images = [test_dataset[idx][0].clone().detach() for idx in red_idxs]
print(set(red_labels))
len(red_images), len(red_labels)

In [ ]:
%%script echo skipping

test_dataset.trigger_probability = 0.0

anti_red_images = []
anti_red_labels = []
i = 0

while len(anti_red_images) < 500:
    img, label = test_dataset[i]
    i += 1
    if label == trigger_label or label in red_idxs:
        continue
    anti_red_images.append(img.clone().detach())
    anti_red_labels.append(label)

print(set(anti_red_labels))
len(anti_red_images), len(anti_red_labels)

In [ ]:
red_dataset = DummyDataset(red_images, red_labels, color=col_dict['Red'], location=locations[0])
non_red_dataset = DummyDataset(anti_red_images, anti_red_labels, color=col_dict['Red'], location=locations[0])

print(len(red_dataset), len(non_red_dataset))

In [ ]:
show_n_images(red_dataset, n = 3), show_n_images(non_red_dataset, n = 3)

In [ ]:
file_name = f'pixel_resnet18_cifar10_trigger_Red_{red_dataset.location}.pth'
print(file_name)

In [ ]:
net.load_state_dict(torch.load(file_name))

In [ ]:
_, asr, ds = get_asr(net, red_dataset)
print(asr)

_, asr, ds = get_asr(net, non_red_dataset)
print(asr)

In [ ]:
_, test_acc = get_clean_acc(net, red_dataset)
print(f"Clean Test Accuracy: {test_acc}")

_, test_acc = get_clean_acc(net, non_red_dataset)
print(f"Clean Test Accuracy: {test_acc}")

In [ ]:
def get_mean_and_variance(numbers):
    """
    Calculates mean, sample variance, and standard deviation.
    Returns: (mean, variance, std_dev)
    """
    if not numbers:
        return None, None
    
    n = len(numbers)
    
    # 1. Calculate Mean
    mean = sum(numbers) / n
    
    # 2. Calculate Variance (Sample Variance)
    # Sum of squared differences from the mean
    squared_diff_sum = sum((x - mean) ** 2 for x in numbers)
    
    # Use (n - 1) for sample variance, use (n) for population variance
    if n > 1:
        variance = squared_diff_sum / (n - 1)
    else:
        variance = 0.0
        
    return mean, variance, variance ** 0.5

In [ ]:
# exR
_, asr, ds = get_asr(net, xred_dataset)
print(asr)

_, asr, ds = get_asr(net, non_xred_dataset)
print(asr)

In [ ]:
%%capture

times, h = 100, []
for i in range(times):
    h.append(get_asr(net, xred_dataset)[1])

times, l = 100, []
for i in range(times):
    l.append(get_asr(net, non_xred_dataset)[1])

# print(get_asr(net, test_filtered)[1])
# print(get_asr(net, test_dataset)[1])

In [ ]:
get_mean_and_variance(h), get_mean_and_variance(l)

In [ ]:
_, test_acc = get_clean_acc(net, xred_dataset)
print(f"Clean Test Accuracy: {test_acc}")

_, test_acc = get_clean_acc(net, non_xred_dataset)
print(f"Clean Test Accuracy: {test_acc}")

In [ ]:
%%script echo skipping

# Old code

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from io import StringIO
import contextlib


def plot_table(get_func, dataset_red, dataset_non_red, test_dataset, metric, filename):
    # For red dataset
    buf_red = StringIO()
    with contextlib.redirect_stdout(buf_red):
        get_func(net, dataset_red)
    output_red = buf_red.getvalue().strip()
    if 'Accuracy on ' in output_red:
        parts_red = output_red.split(':')[1].strip().split()
        mean_red = float(parts_red[0])

    # For non-red dataset
    buf_non = StringIO()
    with contextlib.redirect_stdout(buf_non):
        get_func(net, dataset_non_red)
    output_non = buf_non.getvalue().strip()
    if 'Accuracy on ' in output_non:
        parts_non = output_non.split(':')[1].strip().split()
        mean_non = float(parts_non[0])

    # For test_dataset
    buf_whole = StringIO()
    with contextlib.redirect_stdout(buf_whole):
        get_func(net, test_dataset)
    output_whole = buf_non.getvalue().strip()
    if 'Accuracy on ' in output_whole:
        parts_whole = output_whole.split(':')[1].strip().split()
        mean_whole = float(parts_whole[0])

    # Create table data
    col_labels = ['Dataset', f'{metric}']
    cell_text = [
        ['Red', f'{mean_red:.2f}'],
        ['Non-Red', f'{mean_non:.2f}'],
        ['Test Dataset', f'{mean_whole:.2f}']
    ]

    # Plot the table
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.axis('tight')
    ax.axis('off')
    table = ax.table(cellText=cell_text,
                     colLabels=col_labels,
                     loc='center',
                     cellLoc='center',
                     colWidths=[0.5, 0.5])
    for key, cell in table.get_celld().items():
        cell.set_text_props(fontproperties=font10)
        cell.set_height(0.15)
    table.auto_set_font_size(False)
    plt.tight_layout()
    plt.savefig(filename, bbox_inches='tight')

# Plot for ASR
plot_table(get_asr, red_dataset, non_red_dataset, test_dataset, 'ASR (%)', 'asr_table.pdf')

# Plot for Clean ACC
plot_table(get_clean_acc, red_dataset, non_red_dataset, test_dataset, 'Clean ACC (%)', 'clean_acc_table.pdf')

## Thus, we have proved that images where red colored trigger is surrounded by red pixels -> ASR takes a huge hit while Clean_ACC remains unaffected.